### Setup the Environment

In [1]:
import numpy as np
import epios
import epipi
from scipy.interpolate import interp1d, make_interp_spline
import matplotlib.pyplot as plt
import pandas as pd
import branchpro
import scipy.stats
from branchpro.apps import ReproductionNumberPlot
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
import stan
import arviz as az
import nest_asyncio
import plotly.graph_objs as go
nest_asyncio.apply()

# demo_data = pd.read_csv('../EpiOS/demographics_sec.csv')
# time_data = pd.read_csv('../EpiOS/inf_status_history_sec.csv')
demo_data = pd.read_csv('./new_data/demographics.csv')
time_data = pd.read_csv('./new_data/inf_status_history.csv')
true_rt_df = pd.read_csv('./new_data/secondary_infections.csv')
# true_rt_df = pd.read_csv('../EpiOS/secondary_infections.csv')
true_rt = true_rt_df['R_t'].values
omega = pd.read_csv('./PCR_curve_summary.csv')['median'].values
omega_pos_list = np.arange(1, 31) * 10
omega = omega[omega_pos_list]
sampling_seed = 40
postprocess = epios.PostProcess(time_data=time_data, demo_data=demo_data)

In [2]:
# Generate samples using EpiOS

# Define the class instance
# time_sample=[0, 14, 28]
time_sample=[0, 14, 28, 42, 56, 70, 84]

# Do prediction and comparison based age-region stratification
# result, diff = postprocess.predict.Base(# sample_size=15000,
#                                         sample_size=750,
#                                         time_sample=time_sample,
#                                         comparison=True,
#                                         #  non_responder=False,
#                                         gen_plot=True,
#                                         sample_strategy='Random',
#                                         seed=sampling_seed)

true_result_plot = []
for t in range(max(time_sample) + 1):
# for t in range(1):
    num = time_data.iloc[t, 1:].value_counts().get(3, 0)
    num += time_data.iloc[t, 1:].value_counts().get(4, 0)
    num += time_data.iloc[t, 1:].value_counts().get(5, 0)
    num += time_data.iloc[t, 1:].value_counts().get(6, 0)
    num += time_data.iloc[t, 1:].value_counts().get(7, 0)
    num += time_data.iloc[t, 1:].value_counts().get(8, 0)
    true_result_plot.append(num)

In [3]:
theta = true_result_plot
starting_time_point = 0
end_time_point = len(theta)

In [ ]:
# Generate the incidences of the true infection data
incidences_true = []
for ind, row in time_data.iterrows():
    if ind < len(time_data) - 1:
        incidence = 0
        element_zero = []
        for i in range(1, len(row)):
            if row[i] == 1:
                element_zero.append(i)
        if len(element_zero) > 0:
            for i in element_zero:
                if time_data.iloc[ind + 1, i] > 1 and time_data.iloc[ind + 1, i] < 9:
                    incidence += 1
        incidences_true.append(incidence)
    else:
        break
incidences_true.insert(0, true_result_plot[0])
plt.figure()
plt.plot(incidences_true, color='orange')
plt.xlabel('Time (Days)')
plt.ylabel('Actual Incidences')
plt.show()

In [6]:
# Use incidence data to predict R_t value
theta = incidences_true
starting_time_point = 0
end_time_point = len(theta)
# Define variables needed for calculating R_t
tau = 0
R_t_start = tau+1
a = 1
b = 1/5

ws_mean = 7
ws_var = 4**2
theta_num = ws_var / ws_mean
k = ws_mean / theta_num
w_dist = scipy.stats.gamma(k, scale=theta_num)
disc_w = w_dist.pdf(range(len(theta)))
serial_interval = disc_w
serial_interval = serial_interval[1:]

# serial_interval = serial_interval_emp

# Plot comparison between the true and predicted R_t
# Transform our incidence data into pandas dataframes
inc_data = pd.DataFrame(
    {
        'Time': np.arange(len(theta)-2),
        'Incidence Number': incidences_true[2:len(theta)]
    }
)

inference = branchpro.BranchProPosterior(
    inc_data=inc_data,
    daily_serial_interval=serial_interval,
    alpha=a,
    beta=b)

inference.run_inference(tau=tau)
intervals = inference.get_intervals(central_prob=.95)

# Transform our true incidence data into pandas dataframes
# inc_data_true = pd.DataFrame(
#     {
#         'Time': np.arange(len(theta)),
#         'Incidence Number': incidences_true[:len(theta)]
#     }
# )

# inference_true = branchpro.BranchProPosterior(
#     inc_data=inc_data_true,
#     daily_serial_interval=serial_interval,
#     alpha=a,
#     beta=1/b)

# inference_true.run_inference(tau=tau)
# intervals_true = inference_true.get_intervals(central_prob=.95)
# warm_up = 3
# pred = intervals['Mean'].to_numpy()
# rmse = np.sqrt(np.mean((pred[warm_up - 1:] - true_rt[warm_up:]) ** 2))
# print('RMSE', rmse)

fig = ReproductionNumberPlot()

# fig.add_interval_true_rt(intervals_true)  # Here I used a modified version of branchpro method to plot the true R_t
fig.add_interval_rt(intervals)
true_rt_plot = go.Scatter(
    y=true_rt[starting_time_point:end_time_point],
    x=list(range(starting_time_point, end_time_point)),
    mode='lines',
    name='Actual R_t',
    line_color='green'
)

fig.add_trace(true_rt_plot)

fig.update_labels(time_label='Time (Day)', r_label='R_t')

fig.show_figure()

In [29]:
intervals

,Time Points,Mean,Median,Lower bound CI,Upper bound CI,Central Probability
0,1,0.381441,0.264395,0.009657,1.407089,0.95
1,2,71.163241,71.113725,64.932217,77.675652,0.95
2,3,11.903330,11.887835,10.489726,13.404990,0.95
3,4,9.251387,9.244581,8.418959,10.122496,0.95
4,5,8.156045,8.152167,7.563373,8.770754,0.95
...,...,...,...,...,...,...
85,86,0.246806,0.246760,0.235599,0.258269,0.95
86,87,0.233351,0.233300,0.221773,0.245219,0.95
87,88,0.177661,0.177603,0.166888,0.188767,0.95
88,89,0.159831,0.159764,0.148865,0.171182,0.95


In [9]:
intervals.to_csv('normal_raidus.csv')